# Weather Forecast Silver Pipeline

Điểm vào sản xuất cho pipeline Dự báo Thời tiết Bronze → Silver theo từng giai đoạn.

Notebook này thực hiện:

Khám phá Bronze đã cam kết
→ Phát hiện Silver đã xử lý
→ Chọn các đơn vị chờ nhập
→ Tải Bronze
→ Chuyển đổi
→ Chất lượng dữ liệu
→ Xác thực
→ Lưu trữ Delta

Nếu không có đơn vị chờ nhập nào, pipeline sẽ hoàn thành thành công với `NO_OP`.

In [0]:
import os

account_name = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")
tenant_id = os.getenv("AZURE_TENANT_ID")
client_id = os.getenv("AZURE_CLIENT_ID")
client_secret = os.getenv("AZURE_CLIENT_SECRET")

endpoint = f"{account_name}.dfs.core.windows.net"

spark.conf.set(
    f"fs.azure.account.auth.type.{endpoint}",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{endpoint}",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{endpoint}",
    client_id
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{endpoint}",
    client_secret
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{endpoint}",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

In [0]:
import importlib
import fastorder.transformation.silver.weather.forecast_hourly as forecast_hourly

importlib.reload(forecast_hourly)

In [0]:
from fastorder.transformation.silver.weather.forecast_hourly import (
    discover_committed_forecast_ingestions,
    get_processed_forecast_ingestion_ids,
    find_pending_forecast_ingestions,
    load_pending_forecast_bronze,
    transform_forecast_hourly,
    profile_forecast_data_quality,
    assert_forecast_data_quality,
    profile_forecast_validation,
    assert_forecast_validation,
    prepare_forecast_silver_output,
    write_forecast_silver,
    extract_ingestion_id_from_path,
    run_weather_forecast_silver_pipeline
)

from fastorder.storage.adls_client import (
    get_adls_service_client,
)

In [0]:
BRONZE_FORECAST_ROOT = "weather/open_meteo/forecast"

BRONZE_ABFSS_ROOT = (
    "abfss://bronze@fastorderdatalake.dfs.core.windows.net"
)
SILVER_FORECAST_PATH = (
    "abfss://silver@fastorderdatalake.dfs.core.windows.net/"
    "weather/forecast_hourly/"
)

In [0]:
service_client = get_adls_service_client()

bronze_client = (
    service_client
    .get_file_system_client("bronze")
)

In [0]:
pipeline_result = run_weather_forecast_silver_pipeline(
    spark=spark,
    bronze_client=bronze_client,
    forecast_root=BRONZE_FORECAST_ROOT,
    bronze_abfss_root=BRONZE_ABFSS_ROOT,
    silver_path=SILVER_FORECAST_PATH,
)

pipeline_result